Notebook 

In [1]:
import os

print("Dossier courant :", os.getcwd())
print("Fichiers disponibles :")

for root, dirs, files in os.walk("."):
    dirs[:] = [d for d in dirs if d not in [".git", ".venv", "__pycache__"]]
    for file in files:
        if file.endswith(".parquet"):
            print(os.path.join(root, file))

Dossier courant : /home/jovyan/work
Fichiers disponibles :
./data/instacart/aisles.parquet
./data/instacart/departments.parquet
./data/instacart/orders.parquet
./data/instacart/order_products__prior.parquet
./data/instacart/order_products__train.parquet
./data/instacart/products.parquet
./data/openfoodfacts/produits.parquet
./data/openprices/page.parquet
./data/rappelconso/rappels.parquet


In [2]:
%pip install boto3 python-dotenv pyarrow pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.5/16.5 MB 11.6 MB/s  0:00:01m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [boto3]32m4/5 [boto3]re]
Note: you may need to restart the kernel to use updated packages.


In [3]:
import os

print([
    key for key in os.environ
    if "RUSTFS" in key.upper()
])

[]


In [4]:
import os

print("Dossier courant :", os.getcwd())
print("Fichiers présents :", os.listdir("."))

Dossier courant : /home/jovyan/work
Fichiers présents : ['.env', '.env.example', '.git', '.gitignore', '.ipynb_checkpoints', 'compose.yaml', 'CONTRIBUTING.md', 'data', 'Dockerfile', 'importation', 'README.md', 'script', 'Untitled.ipynb', 'utils']


In [5]:
from pathlib import Path

env_file = Path(".env")

print("Fichier .env présent :", env_file.exists())

Fichier .env présent : True


In [6]:
from dotenv import load_dotenv
import os

load_dotenv(dotenv_path=".env", override=True)

for key, value in os.environ.items():
    if "RUSTFS" in key.upper():
        print(key, "=", "***" if any(x in key.upper() for x in ["SECRET", "PASSWORD", "KEY"]) else value)

In [7]:
import os

print("Endpoint :", os.getenv("S3_ENDPOINT"))
print("Access Key :", os.getenv("S3_ACCESS_KEY"))
print("Secret Key configurée :", bool(os.getenv("S3_SECRET_KEY")))

Endpoint : http://rustfs:9000
Access Key : rustfsadmin
Secret Key configurée : True


In [8]:
%pip install boto3 pyarrow

Note: you may need to restart the kernel to use updated packages.


In [2]:
import boto3
from botocore.config import Config
import os 

s3 = boto3.client(
    "s3",
    endpoint_url=os.getenv("S3_ENDPOINT"),
    aws_access_key_id=os.getenv("S3_ACCESS_KEY"),
    aws_secret_access_key=os.getenv("S3_SECRET_KEY"),
    region_name="us-east-1",
    config=Config(
        signature_version="s3v4",
        s3={"addressing_style": "path"}
    )
)

print("Connexion à RustFS réussie !")

Connexion à RustFS réussie !


In [3]:
response = s3.list_buckets()

print("Buckets disponibles :")

for bucket in response["Buckets"]:
    print("-", bucket["Name"])

Buckets disponibles :


In [4]:
BUCKET_NAME = "bronze"

existing_buckets = [
    bucket["Name"]
    for bucket in s3.list_buckets()["Buckets"]
]

if BUCKET_NAME not in existing_buckets:
    s3.create_bucket(Bucket=BUCKET_NAME)
    print("Bucket bronze créé avec succès.")
else:
    print("Le bucket bronze existe déjà.")

Bucket bronze créé avec succès.


In [8]:
from pathlib import Path

DATA_DIR = Path("/home/jovyan/work")

parquet_files = list(DATA_DIR.rglob("*.parquet"))

print("Nombre de fichiers :", len(parquet_files))

for file in parquet_files:
    print(file.relative_to(DATA_DIR))

Nombre de fichiers : 9
data/instacart/aisles.parquet
data/instacart/departments.parquet
data/instacart/orders.parquet
data/instacart/order_products__prior.parquet
data/instacart/order_products__train.parquet
data/instacart/products.parquet
data/openfoodfacts/produits.parquet
data/openprices/page.parquet
data/rappelconso/rappels.parquet


In [9]:
SOURCES = {
    "instacart": ["instacart"],
    "openfoodfacts": ["openfoodfacts", "open_food_facts"],
    "openprices": ["openprices", "open_prices"],
    "rappelconso": ["rappelconso", "rappel_conso"]
}

def detect_source(path):
    parts = [p.lower() for p in path.parts]

    for source, keywords in SOURCES.items():
        if any(keyword in part for keyword in keywords for part in parts):
            return source

    return None

for file in parquet_files:
    print(detect_source(file), "->", file.name)

instacart -> aisles.parquet
instacart -> departments.parquet
instacart -> orders.parquet
instacart -> order_products__prior.parquet
instacart -> order_products__train.parquet
instacart -> products.parquet
openfoodfacts -> produits.parquet
openprices -> page.parquet
rappelconso -> rappels.parquet


In [10]:
uploaded = 0
unidentified = []

for file in parquet_files:

    source = detect_source(file)

    if source is None:
        unidentified.append(str(file))
        continue

    relative_path = file.relative_to(DATA_DIR)

    # Conserver l'arborescence relative dans chaque source
    object_key = f"{source}/raw/{file.name}"

    s3.upload_file(
        str(file),
        BUCKET_NAME,
        object_key
    )

    print("Importé :", object_key)
    uploaded += 1

print("\nImportation terminée")
print("Fichiers envoyés :", uploaded)
print("Fichiers non identifiés :", len(unidentified))

for file in unidentified:
    print("À classer :", file)

Importé : instacart/raw/aisles.parquet
Importé : instacart/raw/departments.parquet
Importé : instacart/raw/orders.parquet
Importé : instacart/raw/order_products__prior.parquet
Importé : instacart/raw/order_products__train.parquet
Importé : instacart/raw/products.parquet
Importé : openfoodfacts/raw/produits.parquet
Importé : openprices/raw/page.parquet
Importé : rappelconso/raw/rappels.parquet

Importation terminée
Fichiers envoyés : 9
Fichiers non identifiés : 0


In [11]:
paginator = s3.get_paginator("list_objects_v2")

for page in paginator.paginate(Bucket=BUCKET_NAME):
    for obj in page.get("Contents", []):
        print(
            obj["Key"],
            "|",
            round(obj["Size"] / (1024 * 1024), 2),
            "Mo"
        )

instacart/raw/aisles.parquet | 0.0 Mo
instacart/raw/departments.parquet | 0.0 Mo
instacart/raw/order_products__prior.parquet | 133.87 Mo
instacart/raw/order_products__train.parquet | 5.73 Mo
instacart/raw/orders.parquet | 36.66 Mo
instacart/raw/products.parquet | 1.47 Mo
openfoodfacts/raw/produits.parquet | 0.1 Mo
openprices/raw/page.parquet | 0.0 Mo
rappelconso/raw/rappels.parquet | 6.23 Mo


In [12]:
import boto3
import pandas as pd
import pyarrow.parquet as pq
from io import BytesIO

BUCKET_BRONZE = "bronze"
BUCKET_SILVER = "silver"

# Lister les fichiers présents dans Bronze
paginator = s3.get_paginator("list_objects_v2")

bronze_files = []

for page in paginator.paginate(Bucket=BUCKET_BRONZE):
    for obj in page.get("Contents", []):
        if obj["Key"].endswith(".parquet"):
            bronze_files.append({
                "fichier": obj["Key"],
                "taille_Mo": round(obj["Size"] / 1024**2, 2)
            })

df_bronze = pd.DataFrame(bronze_files)

display(df_bronze)

,fichier,taille_Mo
0,instacart/raw/aisles.parquet,0.00
1,instacart/raw/departments.parquet,0.00
2,instacart/raw/order_products__prior.parquet,133.87
3,instacart/raw/order_products__train.parquet,5.73
4,instacart/raw/orders.parquet,36.66
5,instacart/raw/products.parquet,1.47
6,openfoodfacts/raw/produits.parquet,0.10
7,openprices/raw/page.parquet,0.00
8,rappelconso/raw/rappels.parquet,6.23


In [13]:
for file in bronze_files:

    key = file["fichier"]

    print("\n" + "=" * 70)
    print("FICHIER :", key)

    response = s3.get_object(
        Bucket=BUCKET_BRONZE,
        Key=key
    )

    parquet_file = pq.ParquetFile(
        BytesIO(response["Body"].read())
    )

    print("Nombre de lignes :", parquet_file.metadata.num_rows)
    print("Nombre de colonnes :", parquet_file.metadata.num_columns)

    print("\nSTRUCTURE :")
    print(parquet_file.schema)


FICHIER : instacart/raw/aisles.parquet
Nombre de lignes : 134
Nombre de colonnes : 2

STRUCTURE :
required group field_id=-1 schema {
  optional binary field_id=-1 aisle_id (String);
  optional binary field_id=-1 aisle (String);
}


FICHIER : instacart/raw/departments.parquet
Nombre de lignes : 21
Nombre de colonnes : 2

STRUCTURE :
required group field_id=-1 schema {
  optional binary field_id=-1 department_id (String);
  optional binary field_id=-1 department (String);
}


FICHIER : instacart/raw/order_products__prior.parquet
Nombre de lignes : 32434489
Nombre de colonnes : 4

STRUCTURE :
required group field_id=-1 schema {
  optional binary field_id=-1 order_id (String);
  optional binary field_id=-1 product_id (String);
  optional binary field_id=-1 add_to_cart_order (String);
  optional binary field_id=-1 reordered (String);
}


FICHIER : instacart/raw/order_products__train.parquet
Nombre de lignes : 1384617
Nombre de colonnes : 4

STRUCTURE :
required group field_id=-1 schema {


In [14]:
%pip install duckdb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.4/24.4 MB 13.0 MB/s  0:00:01m0:00:0100:01
Note: you may need to restart the kernel to use updated packages.
